<a href="https://colab.research.google.com/github/kazumah1/single-word-decoding/blob/multi-timescale-CNN/colab_trainer_curr_error.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MEG Sentence Decoding — `multi-timescale-CNN` branch

**One-time setup (do this before running):**
1. Open the shared Drive folder: https://drive.google.com/drive/folders/1HOvzJEv0Czn-yJKELq6kp5cORJ5yBTdM
2. Right-click it → **Organize** → **Add shortcut to Drive** → place it in **My Drive** and name it exactly `single-word-decoding`

**Each session:**
1. Run **Cell 1** (mount Drive)
2. Run **Cell 2** (clone `multi-timescale-CNN`, install deps, sync data — ~10 min)
3. Run **Cell 3** (train)
4. Run **Cell 4** (check results)

**Data & results strategy:**
- All branches share the same Drive data folder (no re-downloading across branches)
- Checkpoints and results for this branch go to `single-word-decoding/results/multi-timescale-CNN/`


## Cell 1 — Mount Drive
Run this first. It will ask you to sign in and authorize access.

In [7]:
from google.colab import drive, userdata
from huggingface_hub import login
import os, sys, shutil, torch
drive.mount('/content/drive')

# Auth HuggingFace for llama
token = userdata.get('HF_TOKEN')
login(token)


# ── Branch ─────────────────────────────────────────────────────────────
BRANCH         = 'multi-timescale-CNN'

# ── Paths ──────────────────────────────────────────────────────────────
TRAIN_SUBJECTS = ['sub-01','sub-02','sub-03','sub-04','sub-05','sub-06','sub-07','sub-08','sub-09','sub-10',
                  'sub-11','sub-12','sub-13','sub-14','sub-15','sub-16','sub-17','sub-18','sub-19','sub-20',
                  'sub-21','sub-22','sub-23','sub-24','sub-25','sub-26','sub-27']

REPO           = 'https://github.com/kazumah1/single-word-decoding.git'
WORKDIR        = f'/content/single-word-decoding-{BRANCH.replace("/", "-")}'
LOCAL_DATAPATH = '/content/neural_data'

# Where your data already lives on Drive
DRIVE_GW       = '/content/drive/MyDrive/datasets/gwilliams2022/download'

# Where results/checkpoints will be saved (per-branch)
SAVEPATH       = f'/content/drive/MyDrive/sentence-results/{BRANCH.replace("/", "-")}'
# ───────────────────────────────────────────────────────────────────────

os.makedirs(f'{SAVEPATH}/cache', exist_ok=True)
os.makedirs(LOCAL_DATAPATH,        exist_ok=True)

print(f"Branch: {BRANCH}")
print(f"GPU:    {torch.cuda.get_device_name(0)}")
total, _, free = shutil.disk_usage('/content')
print(f"Disk:   {free/1e9:.0f} GB free / {total/1e9:.0f} GB total")
print(f"\nTrain subjects: {TRAIN_SUBJECTS}")
print(f"Results path:   {SAVEPATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Branch: multi-timescale-CNN
GPU:    NVIDIA A100-SXM4-80GB
Disk:   189 GB free / 253 GB total

Train subjects: ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27']
Results path:   /content/drive/MyDrive/sentence-results/multi-timescale-CNN


## Cell 2 — Full Setup (run once per session, then walk away)
Clones the `multi-timescale-CNN` branch, installs all dependencies, and gets data onto local disk.

In [14]:
import os, shutil, subprocess, sys

# ── 1. Clone repo ──────────────────────────────────────────────────────
print(f'=== Cloning branch: {BRANCH} ===')
if os.path.exists(WORKDIR):
    print('  Already cloned, pulling latest...')
    subprocess.run(['git', '-C', WORKDIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
if not os.path.exists(f'{WORKDIR}/sentence_results'):
    os.symlink(SAVEPATH, f'{WORKDIR}/sentence_results')
os.makedirs(f'{WORKDIR}/projects', exist_ok=True)
print('  Done.')

# ── 2. Install local packages ──────────────────────────────────────────
print('\n=== Installing local packages ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuralset/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuraltrain/'], check=True)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
print('  Done.')

# ── 3. Install dependencies ────────────────────────────────────────────
print('\n=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightning', 'pytorch-lightning', 'torchvision',
                'wandb', 'osfclient', 'mne_bids', 'tqdm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'x-transformers==1.26.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics==1.5.2'], check=True)
print('  Done.')

# ── 4. kenlm ──────────────────────────────────────────────────────────
print('\n=== Installing kenlm ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kenlm'], capture_output=True)
try:
    import kenlm; print('  Installed from PyPI.')
except ImportError:
    print('  PyPI failed — building from source...')
    subprocess.run(['git', 'clone', 'https://github.com/kpu/kenlm', '/content/kenlm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cython'], check=True)
    subprocess.run(['cython', '/content/kenlm/python/kenlm.pyx', '--cplus'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '/content/kenlm'], check=True)
    import kenlm; print('  Built from source.')

# ── 5. Copy training subjects from Drive to local SSD ─────────────────
print('\n=== Copying data from Drive to local disk ===')
local_gw = f'{LOCAL_DATAPATH}/gwilliams2022/download'
os.makedirs(local_gw, exist_ok=True)

for item in ['stimuli'] + TRAIN_SUBJECTS:
    src = f'{DRIVE_GW}/{item}'
    dst = f'{local_gw}/{item}'
    if os.path.exists(dst):
        print(f'  {item} already on local disk, skipping.')
    elif os.path.exists(src):
        print(f'  Copying {item}...')
        shutil.copytree(src, dst)
    else:
        print(f'  WARNING: {item} not found at {src}')

_, _, free = shutil.disk_usage('/content')
print(f'  Disk free: {free/1e9:.0f} GB')
print(f'\n✓ Setup complete. Run Cell 3 to train on branch: {BRANCH}')

=== Cloning branch: multi-timescale-CNN ===
  Already cloned, pulling latest...
  Done.

=== Installing local packages ===
  Done.

=== Installing dependencies ===
  Done.

=== Installing kenlm ===
  Installed from PyPI.

=== Copying data from Drive to local disk ===
  stimuli already on local disk, skipping.
  sub-01 already on local disk, skipping.
  sub-02 already on local disk, skipping.
  sub-03 already on local disk, skipping.
  sub-04 already on local disk, skipping.
  sub-05 already on local disk, skipping.
  sub-06 already on local disk, skipping.
  sub-07 already on local disk, skipping.
  sub-08 already on local disk, skipping.
  sub-09 already on local disk, skipping.
  sub-10 already on local disk, skipping.
  sub-11 already on local disk, skipping.
  sub-12 already on local disk, skipping.
  sub-13 already on local disk, skipping.
  sub-14 already on local disk, skipping.
  sub-15 already on local disk, skipping.
  sub-16 already on local disk, skipping.
  sub-17 already 

In [15]:
import re

path = f'{WORKDIR}/neuraltrain/neuraltrain/models/multiscaleconv.py'
with open(path) as f:
    src = f.read()

# 1. Add imports from simpleconv after the transformer import line
src = src.replace(
    'from .transformer import LlamaTransformerConfig, TransformerEncoderConfig',
    'from .transformer import LlamaTransformerConfig, TransformerEncoderConfig\n'
    'from .simpleconv import (\n'
    '    ConvSequence, SpatialFilter,\n'
    '    SimpleConvConfig, SimpleConv,\n'
    '    SimpleConvTimeAggConfig, SimpleConvTimeAgg,\n'
    ')'
)

# 2. Remove the duplicate class blocks for SimpleConvConfig, SimpleConv,
#    SimpleConvTimeAggConfig, SimpleConvTimeAgg — everything between
#    MultiScaleConvSequence and MultiScaleSimpleConvConfig
src = re.sub(
    r'(\n# -{5,}\n# MultiScaleConvSequence.*?)\n# -{5,}\n# SimpleConvConfig.*?\n# -{5,}\n# MultiScaleSimpleConvConfig',
    r'\1\n\n# ---------------------------------------------------------------------------\n# MultiScaleSimpleConvConfig',
    src, flags=re.DOTALL
)

# 3. Remove duplicate SimpleConvTimeAggConfig and SimpleConvTimeAgg blocks
#    (between MultiScaleSimpleConv and MultiScaleSimpleConvTimeAgg)
src = re.sub(
    r'(\nclass MultiScaleSimpleConv\b.*?)\nclass SimpleConvTimeAggConfig\b.*?\nclass MultiScaleSimpleConvTimeAggConfig',
    r'\1\n\nclass MultiScaleSimpleConvTimeAggConfig',
    src, flags=re.DOTALL
)

with open(path, 'w') as f:
    f.write(src)

# Verify
classes = [l for l in src.splitlines() if l.startswith('class ')]
print('Classes in file:')
for c in classes:
    print(' ', c)


Classes in file:
  class SpatialFilter(nn.Module):
  class ConvSequence(nn.Module):
  class MultiScaleConvSequence(nn.Module):
  class MultiScaleSimpleConvConfig(SimpleConvConfig):
  class SimpleConv(nn.Module):
  class MultiScaleSimpleConv(SimpleConv):
  class MultiScaleSimpleConvTimeAggConfig(MultiScaleSimpleConvConfig):
  class MultiScaleSimpleConvTimeAgg(MultiScaleSimpleConv):


In [16]:
path = f'{WORKDIR}/sentence_decoding/grids/test.py'
with open(path) as f:
    src = f.read()

src = src.replace('"data.n_timelines": 1', '"data.n_timelines": 10000')
src = src.replace('"save_checkpoints": False', '"save_checkpoints": True')

with open(path, 'w') as f:
    f.write(src)
print('Patched. Now run Cell 3.')

Patched. Now run Cell 3.


In [21]:
path = f'{WORKDIR}/sentence_decoding/grids/test.py'
with open(path) as f:
    src = f.read()

src = src.replace('"data.n_timelines": 1', '"data.n_timelines": 10000')
src = src.replace('"save_checkpoints": False', '"save_checkpoints": True')

with open(path, 'w') as f:
    f.write(src)
print('n_timelines:', '10000' if '"data.n_timelines": 10000' in src else 'FAILED')
print('save_checkpoints:', 'True' if '"save_checkpoints": True' in src else 'FAILED')

n_timelines: 10000
save_checkpoints: True


In [20]:
import shutil
path = f'{SAVEPATH}/cache/sentence_decoding/StudyLoader,0-v2,Gwilliams2022/neuralset.data.StudyLoader._build,ntimelines=10000,name=Gwilliams2022-9f9397f9'
shutil.rmtree(path)
print('Deleted')


Deleted


## Cell 3 — Train

In [22]:
import os
os.environ['DATAPATH']   = LOCAL_DATAPATH
os.environ['SAVEPATH']   = SAVEPATH
os.environ['WANDB_MODE'] = 'online'  # set to 'online' to enable wandb
os.environ['PYTHONPATH'] = WORKDIR

!DATAPATH={LOCAL_DATAPATH} SAVEPATH={SAVEPATH} WANDB_MODE=disabled PYTHONPATH={WORKDIR} \
    python -m sentence_decoding.grids.test


2026-04-16 22:04:20 - WARNING - neuralset.infra.utils:212 - Did not find a discriminator for transformer_config (uid may be incomplete)
2026-04-16 22:04:20 - WARNING - neuralset.infra.task:79 - Cached computation failed with traceback:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numexpr/necompiler.py", line 905, in validate
    compiled_ex = _numexpr_cache.c[numexpr_key]
                  ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
KeyError: ('(index) < (10000000000000000000000000000)', (('optimization', 'aggressive'), ('truediv', False)), (('index', <class 'numpy.int64'>),))

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/single-word-decoding-multi-timescale-CNN/neuralset/build/__editable__.neuralset-0.0.1-py3-none-any/neuralset/infra/task.py", line 49, in __init__
    out = func()
          ^^^^^^
  File "/content/single-word-decoding-multi-timescale-CNN/sentence_decoding/main.py", line

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
result = subprocess.run(['find', SAVEPATH, '-name', '*.ckpt'], capture_output=True, text=True)
print(result.stdout)

## Cell 4 — Check Results

In [ ]:
import glob, json, os

result_dirs = sorted(glob.glob(f'{SAVEPATH}/results/sentence_decoding/*'))
if not result_dirs:
    # Fallback: search one level up
    result_dirs = sorted(glob.glob(f'{SAVEPATH}/**/*', recursive=False))
if not result_dirs:
    print('No results yet.')
else:
    latest = result_dirs[-1]
    print(f'Branch:     {BRANCH}')
    print(f'Latest run: {latest}')
    for f in sorted(os.listdir(latest)):
        print(f'  {f}')

    metrics_path = os.path.join(latest, 'metrics.json')
    if os.path.exists(metrics_path):
        with open(metrics_path) as fh:
            metrics = json.load(fh)
        print('\nMetrics:')
        for k, v in metrics.items():
            print(f'  {k}: {v}')


Branch:     multi-timescale-CNN
Latest run: /content/drive/MyDrive/sentence-results/multi-timescale-CNN/results/sentence_decoding/test
  data.dataset=Gwilliams2022


In [ ]:
print(WORKDIR)


/content/single-word-decoding-multi-timescale-CNN
